In [42]:
# ==============================================================================
# CELL 1: Setup and Imports
# ==============================================================================
"""
Copy this entire section into your first Jupyter cell:
"""

# Import required libraries
import os
import sys
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Add project paths - Jupyter notebook version
# Get the current working directory (where the notebook is located)
current_dir = os.getcwd()
src_path = os.path.join(current_dir, "src")
sys.path.insert(0, src_path)

# Add ODYM framework path (adjust as needed)
project_root_parent = os.path.dirname(current_dir)
odym_path = os.path.join(
    project_root_parent, "framework", "ODYM-master_20241127", "odym", "modules"
)
sys.path.insert(0, odym_path)

# Import BioDYM modules
try:
    import config
    import data_loader
    import system_setup
    import utils
    from engine import solver
    import plotting
    import ODYM_Classes as msc

    print("✅ All modules imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please check your installation and paths.")
    raise

✅ All modules imported successfully!


In [44]:
import os
import sys

# Add project and ODYM framework paths
current_dir = os.getcwd()
src_path = os.path.join(current_dir, "src")
sys.path.insert(0, src_path)
project_root_parent = os.path.dirname(current_dir)
odym_path = os.path.join(
    project_root_parent, "framework", "ODYM-master_20241127", "odym", "modules"
)
sys.path.insert(0, odym_path)

# Import BioDYM modules
import config
import data_loader
import system_setup
from engine import solver
import plotting

# Set path to golden dataset
golden_path = "test_data/golden_dataset.xlsx"


# --- Define config as a class instance ---
class AnalysisConfig:
    def __init__(self):
        self.excel_file_path = golden_path
        self.output_path = "data/02_output/results.xlsx"
        self.start_year = 2025
        self.end_year = 2030
        self.elements = ["material", "WC", "DM", "CC"]
        self.run_monte_carlo = False
        self.mc_iterations = 100
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


config = AnalysisConfig()
print("✅ Config loaded:", vars(config))

# Load and validate data
input_data = pd.read_excel(
    config.excel_file_path,
    sheet_name=None,
    header=0,
    engine="openpyxl",
    na_values=["N.A.", "NA", "n/a"],
)
data_loader.validate_input_data(input_data)
print("✅ Data validation passed!")

# Model setup
model_classification, index_table = system_setup.define_model_scope(
    config.start_year, config.end_year, config.elements
)
mfa_system_base = system_setup.initialize_mfa_system(model_classification, index_table)
mfa_system_base, all_excel_data = system_setup.load_and_define_processes(
    mfa_system_base, config.excel_file_path, data_loader
)
mfa_system_configured, all_excel_data = system_setup.define_flows_and_parameters(
    mfa_system_base, all_excel_data
)
dsm_params = data_loader.load_dsm_parameters(all_excel_data)
fomp_params = data_loader.load_fomp_parameters(all_excel_data)
uncertainty_params = data_loader.load_uncertainty_definitions(all_excel_data)

# Run calculation
mfa_system_with_results, dsm_details = solver.run_mfa_calculation(
    mfa_system_configured, dsm_params, fomp_params, config
)
print("✅ Calculation complete!")

# --- Visualization ---
print("📊 Mass Balance Error Plot")
plotting.plot_mass_balance_error(mfa_system_with_results)

print("📊 Individual Flows")
plotting.plot_individual_flows(mfa_system_with_results)

print("📈 Individual Stocks")
plotting.plot_individual_stocks(mfa_system_with_results, dsm_params, fomp_params)

print("🌊 Sankey Diagram")
plotting.plot_interactive_sankey(mfa_system_with_results)

✅ Config loaded: {'excel_file_path': 'test_data/golden_dataset.xlsx', 'output_path': 'data/02_output/results.xlsx', 'start_year': 2025, 'end_year': 2030, 'elements': ['material', 'WC', 'DM', 'CC'], 'run_monte_carlo': False, 'mc_iterations': 100, 'RUN_DSM_CALCULATION': False, 'RUN_FOMP_CALCULATION': True}
--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
✅ Data validation passed!
--> Model scope and classifications defined.
--> MFA system object initialized.
--> Defining process and stock structures...
--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
--> Stock values initialized.
--> Defining flows, parameters, and setting all initial values...
--> All flows initialized to zero.
--> Populated data for primary input flows.
--> Loading DSM parameters from sheet '3_1_Definition_DSM'...
--> Successfully loaded configurations for 0 DSM process(es).


interactive(children=(IntSlider(value=2025, description='Year', max=2030, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'marker': {'color': ['#2ca02c', '#7f7f7f', '#d62728']},
              'type': 'bar',
              'uid': '4172b83a-9dc5-4997-979d-58fbe3ce2ca9',
              'x': [Atmosphere, Environment, Lithosphere],
              'y': [-80.0, 0.0, 80.0]}],
    'layout': {'height': 500,
               'shapes': [{'line': {'color': 'black', 'width': 2},
                           'type': 'line',
                           'x0': -0.5,
                           'x1': 2.5,
                           'y0': 0,
                           'y1': 0}],
               'template': '...',
               'title': {'text': 'Mass Balance Error Check for MATERIAL in 2025'},
               'yaxis': {'title': {'text': 'Error in Mg (positive = mass created)'}}}
})

📊 Individual Flows


interactive(children=(SelectMultiple(description='Select Flows:', index=(0,), options=('F_00_01', 'F_01_00', '…

...

interactive(children=(SelectMultiple(description='Select Stocks:', index=(0,), options=('Environment',), rows=…

FigureWidget({
    'data': [], 'layout': {'template': '...'}
})

🌊 Sankey Diagram


interactive(children=(IntSlider(value=2025, description='Year', max=2030, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'link': {'source': [0, 1, 1], 'target': [1, 2, 0], 'value': [100.0, 80.0, 20.0]},
              'node': {'color': 'blue', 'label': ['Atmosphere', 'Environment', 'Lithosphere']},
              'type': 'sankey',
              'uid': 'a628622b-c7df-4b35-b108-0c3b470beef1'}],
    'layout': {'font': {'size': 12},
               'height': 700,
               'margin': {'b': 20, 'l': 10, 'r': 10, 't': 50},
               'template': '...',
               'title': {'text': 'MFA Sankey for MATERIAL in 2025 (Flows > 0.00 Mg)'}}
})

In [ ]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [ ]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [4]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [5]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [6]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [7]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [8]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [9]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [10]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [11]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [12]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [13]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [14]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [15]:
# ==============================================================================
# CELL 3: Data Validation
# ==============================================================================
"""
Copy this entire section into your third Jupyter cell:
"""

# Check input file
if os.path.exists(config.excel_file_path):
    print(f"✅ Input file found: {config.excel_file_path}")

    # Load and validate data
    try:
        input_data = pd.read_excel(
            config.excel_file_path,
            sheet_name=None,
            header=0,
            engine="openpyxl",
            na_values=["N.A.", "NA", "n/a"],
        )
        data_loader.validate_input_data(input_data)
        print("✅ Data validation passed!")

        # Show available sheets
        print(f"\n📊 Available data sheets: {list(input_data.keys())}")

        # Show basic info
        if "0_Metadata" in input_data:
            metadata = input_data["0_Metadata"]
            print(
                f"\n📋 Dataset: {metadata.get('Dataset_Name', 'Unknown').iloc[0] if len(metadata) > 0 else 'Unknown'}"
            )
            print(
                f"   Version: {metadata.get('Version', 'Unknown').iloc[0] if len(metadata) > 0 else 'Unknown'}"
            )
            print(
                f"   Author: {metadata.get('Author', 'Unknown').iloc[0] if len(metadata) > 0 else 'Unknown'}"
            )

    except Exception as e:
        print(f"❌ Data validation failed: {e}")
        raise
else:
    print(f"❌ Input file not found: {config.excel_file_path}")
    print("Please check the file path in the configuration above.")

✅ Input file found: data/01_input/250707_Template_CS1.xlsx
--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
✅ Data validation passed!

📊 Available data sheets: ['Version_CS0_07.07.', '0_ReadMe', 'Table of Content', '0_Configuration', '1_1_Definition_Flows', '1_2_Data_Flows', '2_1_Definition_Processes', '2_3_Process_TCs', '2_4_Initial_Stock', '2_5_dynamic_tcs', '3_1_Definition_DSM', '3_2_Definition_FOMP', '4_1_Uncertainty_Parameters', '3. TC_Data', 'PX - Template', '4. Calculation_factors>>>>', '4. Codelists>>>>', '4_1 Codelists', '5. Wastefiles >>>>']


In [ ]:
# ==============================================================================
# CELL 4: Model Setup
# ==============================================================================
"""
Copy this entire section into your fourth Jupyter cell:
"""

# Model setup and execution
print("🔧 Setting up MFA model...")

# 1. Define model scope
model_classification, index_table = system_setup.define_model_scope(
    config.start_year, config.end_year, config.elements
)

# 2. Initialize MFA system
mfa_system_base = system_setup.initialize_mfa_system(model_classification, index_table)

# 3. Load data and define processes
mfa_system_base, all_excel_data = system_setup.load_and_define_processes(
    mfa_system_base, config.excel_file_path, data_loader
)

# 4. Load model parameters
dsm_params = data_loader.load_dsm_parameters(all_excel_data)
fomp_params = data_loader.load_fomp_parameters(all_excel_data)
uncertainty_params = data_loader.load_uncertainty_definitions(all_excel_data)

# 5. Configure system with flows and parameters
mfa_system_configured, _ = system_setup.define_flows_and_parameters(
    mfa_system_base, all_excel_data
)

print("✅ Model setup complete!")
print(f"   Processes: {len(mfa_system_configured.ProcessList)}")
print(f"   Flows: {len(mfa_system_configured.FlowDict)}")
print(f"   Stocks: {len(mfa_system_configured.StockDict)}")
print(f"   Parameters: {len(mfa_system_configured.ParameterDict)}")

🔧 Setting up MFA model...
--> Model scope and classifications defined.
--> MFA system object initialized.
--> Defining process and stock structures...
--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
--> Stock values initialized.
--> Loading DSM parameters from sheet '3_1_Definition_DSM'...
--> Successfully loaded configurations for 0 DSM process(es).
--> Loading FOMP parameters from sheet '3_2_Definition_FOMP'...
--> Successfully loaded configurations for 1 FOMP process(es).
--> Loading uncertainty definitions from sheet '4_1_Uncertainty_Parameters'...
--> Successfully loaded 4 uncertainty parameter definition(s).
--> Defining flows, parameters, and setting all initial values...
--> All flows initialized to zero.
--> Populated data for primary input flows.
✅ Model setup complete!
   Processes: 11
   Flows: 18
   Stocks: 8
   Parameters: 72


In [17]:
# ==============================================================================
# CELL 5: Run Calculations
# ==============================================================================
"""
Copy this entire section into your fifth Jupyter cell:
"""

# Run calculations
print("\n🧮 Running MFA calculations...")

if config.run_monte_carlo:
    print(f"   Monte Carlo simulation ({config.mc_iterations} iterations)")

    # Monte Carlo simulation
    mc_run_results = []

    for i in range(config.mc_iterations):
        if i % 10 == 0:  # Progress indicator
            print(f"   Progress: {i}/{config.mc_iterations}")

        # Sample parameters
        sampled_values = utils.sample_parameters(uncertainty_params)
        tc_updates = {k: v for k, v in sampled_values.items() if k.startswith("TC_")}

        # Run calculation
        run_results, _ = solver.run_mfa_calculation(
            mfa_system_configured,
            dsm_params,
            fomp_params,
            config,
            tc_updates=tc_updates,
        )

        # Extract KPIs
        if run_results:
            final_c_stock_soil = run_results.StockDict["S_8"].Values[-1, 3]
            current_run_data = sampled_values.copy()
            current_run_data["run_id"] = i
            current_run_data["final_C_stock_soil"] = final_c_stock_soil
            mc_run_results.append(current_run_data)

    df_mc_results = pd.DataFrame(mc_run_results)
    mfa_system_with_results = None
    dsm_details = None

    print("✅ Monte Carlo simulation complete!")

else:
    print("   Deterministic calculation")

    # Single deterministic run
    mfa_system_with_results, dsm_details = solver.run_mfa_calculation(
        mfa_system_configured, dsm_params, fomp_params, config
    )

    df_mc_results = None

    print("✅ Deterministic calculation complete!")


🧮 Running MFA calculations...
   Deterministic calculation
--> Calculating final stock balances for ALL processes...
--> Stock balance calculation finished.
✅ Deterministic calculation complete!


In [18]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [19]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [20]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [21]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [22]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [23]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [24]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [25]:
# ==============================================================================
# CELL 2: Data and Configuration Setup
# ==============================================================================
"""
Copy this entire section into your second Jupyter cell:
"""

# Configuration settings


class AnalysisConfig:
    def __init__(self):
        # File paths
        self.excel_file_path = "data/01_input/250707_Template_CS1.xlsx"
        self.output_path = "data/02_output/results.xlsx"

        # Model scope
        self.start_year = 2025
        self.end_year = 2050
        self.elements = ["material", "WC", "DM", "CC"]

        # Calculation options
        self.run_monte_carlo = False
        self.mc_iterations = 100

        # Model components - using uppercase names that solver expects
        self.RUN_DSM_CALCULATION = False
        self.RUN_FOMP_CALCULATION = True


# Create configuration instance
config = AnalysisConfig()

# Validate configuration
print("📋 Analysis Configuration:")
print(f"   Input file: {config.excel_file_path}")
print(f"   Time range: {config.start_year} - {config.end_year}")
print(f"   Elements: {', '.join(config.elements)}")
print(f"   Monte Carlo: {'Yes' if config.run_monte_carlo else 'No'}")
if config.run_monte_carlo:
    print(f"   MC iterations: {config.mc_iterations}")

# Validate configuration
print("\n🔍 Configuration Validation:")
is_valid = utils.print_configuration_summary(config)

if not is_valid:
    print("\n❌ Please fix configuration issues before proceeding.")
    raise ValueError("Configuration validation failed")

📋 Analysis Configuration:
   Input file: data/01_input/250707_Template_CS1.xlsx
   Time range: 2025 - 2050
   Elements: material, WC, DM, CC
   Monte Carlo: No

🔍 Configuration Validation:
📋 Configuration Summary
   ✅ Input file found: data/01_input/250707_Template_CS1.xlsx
   ✅ Time range: 2025 - 2050
   ✅ Elements: material, WC, DM, CC
   ✅ DSM calculation: Disabled
   ✅ FOMP calculation: Enabled
   ✅ Output directory: data/02_output

✅ Configuration is valid and ready for analysis!


In [26]:
# ==============================================================================
# CELL 6: Results Summary
# ==============================================================================
"""
Copy this entire section into your sixth Jupyter cell:
"""

# Display results summary
print("📊 MFA Results Summary")
print("=" * 50)

if config.run_monte_carlo and df_mc_results is not None:
    # Monte Carlo results
    print("\n🎲 Monte Carlo Results:")

    if "final_C_stock_soil" in df_mc_results.columns:
        final_c_stock = df_mc_results["final_C_stock_soil"]
        print("   Final Soil Carbon Stock:")
        print(f"     Mean: {final_c_stock.mean():.2f} Mg C")
        print(f"     5th percentile: {final_c_stock.quantile(0.05):.2f} Mg C")
        print(f"     95th percentile: {final_c_stock.quantile(0.95):.2f} Mg C")
        print(f"     Standard deviation: {final_c_stock.std():.2f} Mg C")

    # Show uncertainty parameters
    uncertainty_cols = [
        col
        for col in df_mc_results.columns
        if col not in ["run_id", "final_C_stock_soil"]
    ]
    if uncertainty_cols:
        print(f"\n📈 Uncertain Parameters: {len(uncertainty_cols)}")
        for param in uncertainty_cols[:5]:  # Show first 5
            print(f"   - {param}")
        if len(uncertainty_cols) > 5:
            print(f"   ... and {len(uncertainty_cols) - 5} more")

else:
    # Deterministic results
    if mfa_system_with_results is not None:
        print("\n📈 Deterministic Results:")

        # Show final stock values
        print("   Final Stock Values (last year):")
        for stock_name, stock_obj in mfa_system_with_results.StockDict.items():
            if stock_name.startswith("S_"):  # Only show actual stocks, not delta stocks
                final_values = stock_obj.Values[-1, :]
                print(f"     {stock_name}: {final_values[0]:.2f} Mg material")

                # Show carbon content if available
                if len(final_values) > 3:
                    print(f"            {final_values[3]:.2f} Mg C")

        # Show total flows
        print("\n   Total Flow Values (last year):")
        total_flows = 0
        for flow_name, flow_obj in mfa_system_with_results.FlowDict.items():
            if np.any(flow_obj.Values[-1, :] > 0):  # Only show non-zero flows
                total_flow = flow_obj.Values[-1, 0]
                total_flows += total_flow
                print(f"     {flow_name}: {total_flow:.2f} Mg material")

        print(f"\n   Total system flows: {total_flows:.2f} Mg material")

    else:
        print("❌ No results available")

print("\n" + "=" * 50)

📊 MFA Results Summary

📈 Deterministic Results:
   Final Stock Values (last year):
     S_0: -129.62 Mg material
            -2714.81 Mg C
     S_1: -795.00 Mg material
            2252.50 Mg C
     S_6: 0.00 Mg material
            0.00 Mg C
     S_10: 924.62 Mg material
            462.31 Mg C

   Total Flow Values (last year):
     F_00_02: 350.00 Mg material
     F_01_02: 350.00 Mg material
     F_02_03: 700.00 Mg material
     F_03_04: 350.00 Mg material
     F_03_05: 350.00 Mg material
     F_04_00: 175.00 Mg material
     F_04_01: 175.00 Mg material
     F_05_06: 140.00 Mg material
     F_06_07: 140.00 Mg material
     F_07_00: 70.00 Mg material
     F_07_01: 70.00 Mg material
     F_05_08: 105.00 Mg material
     F_08_10: 105.00 Mg material
     F_05_09: 105.00 Mg material
     F_09_00: 52.50 Mg material
     F_09_01: 52.50 Mg material
     F_10_00: 51.58 Mg material

   Total system flows: 3341.58 Mg material



In [ ]:
# ==============================================================================
# CELL 7: MASS BALANCE VALIDATION (MOST IMPORTANT - RUN FIRST!)
# ==============================================================================
"""
Copy this entire section into your seventh Jupyter cell:

## 🔍 Mass Balance Validation - CRITICAL FIRST STEP

**Why this comes first:** Mass balance validation is the most important check in MFA analysis. 
It ensures that material conservation is maintained throughout the system. 
- Green bars = balanced processes (good!)
- Red bars = mass created (error!)
- Gray bars = mass destroyed (error!)

**What to look for:** All bars should be close to zero (green). 
If you see red or gray bars, there's an issue with your model setup.
"""

# Mass Balance Check - CRITICAL FIRST STEP
print("🔍 MASS BALANCE VALIDATION - CRITICAL FIRST STEP")
print("=" * 60)
print("This is the MOST IMPORTANT validation step!")
print("Green bars = balanced, Red bars = mass created, Gray bars = mass destroyed")
print("All bars should be close to zero for a valid model.")
print("=" * 60)

if mfa_system_with_results is not None:
    plotting.plot_mass_balance_error(mfa_system_with_results)

    # Additional mass balance summary
    print("\n📊 Mass Balance Summary:")
    time_items = mfa_system_with_results.IndexTable.Classification["Time"].Items
    element_items = mfa_system_with_results.Elements

    # Check final year mass balance
    final_year_idx = -1
    for element in element_items:
        element_index = element_items.index(element)
        total_error = 0

        for p in mfa_system_with_results.ProcessList:
            in_val = sum(
                f.Values[final_year_idx, element_index]
                for f in mfa_system_with_results.FlowDict.values()
                if f.P_End == p.ID
            )
            out_val = sum(
                f.Values[final_year_idx, element_index]
                for f in mfa_system_with_results.FlowDict.values()
                if f.P_Start == p.ID
            )
            ds_val = mfa_system_with_results.StockDict.get(f"dS_{p.ID}", None)
            ds_sum = (
                ds_val.Values[final_year_idx, element_index]
                if ds_val is not None
                else 0
            )
            error = in_val - out_val - ds_sum
            total_error += abs(error)

        print(f"   {element.upper()}: Total absolute error = {total_error:.6f} Mg")

        if total_error < 1e-6:
            print(f"   ✅ {element.upper()} mass balance is excellent!")
        elif total_error < 1e-3:
            print(f"   ⚠️  {element.upper()} mass balance is acceptable.")
        else:
            print(f"   ❌ {element.upper()} mass balance has issues!")
else:
    print("❌ No deterministic results available for mass balance check")

🔍 MASS BALANCE VALIDATION - CRITICAL FIRST STEP
This is the MOST IMPORTANT validation step!
Green bars = balanced, Red bars = mass created, Gray bars = mass destroyed
All bars should be close to zero for a valid model.


interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'marker': {'color': [#7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f,
                                   #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f,
                                   #7f7f7f]},
              'type': 'bar',
              'uid': '15bc225c-5500-43e9-9a41-62eea6fc9934',
              'x': [Atmosphere, Environment, Cultivation, Harvest, Grain
                    Processing & Consumption, Straw d&C, Utilization in
                    construction, Incineration, Incorporation, Animal bedding,
                    Lithosphere],
              'y': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}],
    'layout': {'height': 500,
               'shapes': [{'line': {'color': 'black', 'width': 2},
                           'type': 'line',
                           'x0': -0.5,
                           'x1': 10.5,
                           'y0': 0,
                           'y1': 0}],
               'template': '...',
               'title': {'t


📊 Mass Balance Summary:
   MATERIAL: Total absolute error = 0.000000 Mg
   ✅ MATERIAL mass balance is excellent!
   WC: Total absolute error = 0.000000 Mg
   ✅ WC mass balance is excellent!
   DM: Total absolute error = 0.000000 Mg
   ✅ DM mass balance is excellent!
   CC: Total absolute error = 0.000000 Mg
   ✅ CC mass balance is excellent!


In [ ]:
# ==============================================================================
# CELL 8: INDIVIDUAL FLOW ANALYSIS
# ==============================================================================
"""
Copy this entire section into your eighth Jupyter cell:

## 📊 Individual Flow Analysis

**Purpose:** Analyze specific flows in detail to understand material movements.
**Features:**
- Select multiple flows to compare
- Choose between line and bar charts
- Option to show cumulative values
- Filter by element type

**Use cases:**
- Identify dominant flows in the system
- Track material pathways
- Compare flow magnitudes over time
- Analyze cumulative material movements
"""

# Individual Flow Analysis
print("📊 Individual Flow Analysis")
print("=" * 50)
print("Select specific flows to analyze their time evolution.")
print("Use the dropdown to choose flows and elements.")

if mfa_system_with_results is not None:
    plotting.plot_individual_flows(mfa_system_with_results)
else:
    print("❌ No deterministic results available for flow analysis")

📊 Individual Flow Analysis
Select specific flows to analyze their time evolution.
Use the dropdown to choose flows and elements.


interactive(children=(SelectMultiple(description='Select Flows:', index=(0,), options=('F_00_02', 'F_01_02', '…

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'F_00_02',
              'type': 'scatter',
              'uid': 'b5b519fe-0cdf-46b1-8136-54ef5a534ca9',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'y': array([100., 110., 120., 130., 140., 150., 160., 170., 180., 190., 200.,  10.,
                          220., 230., 240., 250., 260., 270., 280., 290., 300., 310., 320., 330.,
                          340., 350.])}],
    'layout': {'barmode': 'overlay',
               'height': 500,
               'hovermode': 'x unified',
               'template': '...',
               'title': {'text': 'Flow Analysis (MATERIAL)'},
               'xaxis': {'title': {'text': 'Year'}},
               'yaxis': {'title': {'text': 'Mass in Mg'}}}
})

In [ ]:
# ==============================================================================
# CELL 9: INDIVIDUAL STOCK ANALYSIS
# ==============================================================================
"""
Copy this entire section into your ninth Jupyter cell:

## 📈 Individual Stock Analysis

**Purpose:** Analyze specific stocks to understand material accumulation patterns.
**Features:**
- Select multiple stocks to compare
- Color-coded by process type (DSM, FOMP, Regular)
- Option to show stock changes (ΔS) instead of absolute stocks
- Choose between line and bar charts

**Use cases:**
- Track material accumulation in specific processes
- Compare stock evolution between different process types
- Analyze stock change rates
- Identify processes with significant material storage
"""

if mfa_system_with_results is not None:
    # Call your plotting function
    plotting.plot_individual_stocks(mfa_system_with_results)
else:
    print(
        "❌ No deterministic results available for this plot (Monte Carlo mode is active)."
    )
    print(
        "Switch off Monte Carlo (config.run_monte_carlo = False) and re-run calculations for these plots."
    )

# Individual Stock Analysis
print("Available stocks:", list(mfa_system_with_results.StockDict.keys()))
print("DSM params:", dsm_params)
print("FOMP params:", fomp_params)

print("📈 Individual Stock Analysis")
print("=" * 50)
print("Select specific stocks to analyze their time evolution.")
print("DSM processes: Orange dashed lines")
print("FOMP processes: Green dot-dash lines")
print("Regular processes: Blue solid lines")

if mfa_system_with_results is not None:
    plotting.plot_individual_stocks(mfa_system_with_results, dsm_params, fomp_params)
else:
    print("❌ No deterministic results available for stock analysis")

interactive(children=(SelectMultiple(description='Select Stocks:', index=(0,), options=('Atmosphere', 'Environ…

FigureWidget({
    'data': [], 'layout': {'template': '...'}
})

Available stocks: ['dS_0', 'S_0', 'dS_1', 'S_1', 'dS_6', 'S_6', 'dS_10', 'S_10']
DSM params: {}
FOMP params: {10: {'outflow_id': 'F_10_00', 'f': 0.236, 'k1': 0.025, 'k2': 0.0351}}
📈 Individual Stock Analysis
Select specific stocks to analyze their time evolution.
DSM processes: Orange dashed lines
FOMP processes: Green dot-dash lines
Regular processes: Blue solid lines


interactive(children=(SelectMultiple(description='Select Stocks:', index=(0,), options=('Atmosphere', 'Environ…

FigureWidget({
    'data': [], 'layout': {'template': '...'}
})

In [ ]:
# ==============================================================================
# CELL 10: SYSTEM OVERVIEW - SANKEY DIAGRAM
# ==============================================================================
"""
Copy this entire section into your tenth Jupyter cell:

## 🌊 System Overview - Material Flow Network (Sankey)

**Purpose:** Visualize the entire material flow network in an interactive diagram.
**Features:**
- Interactive controls for year, element, and flow threshold
- Filter processes to focus on specific parts of the system
- Adjust minimum flow value to hide minor flows
- Real-time updates

**Use cases:**
- Understand overall system structure
- Identify major material pathways
- Communicate system complexity to stakeholders
- Identify bottlenecks or dominant flows
"""

# Interactive Sankey Diagram
print("🌊 Material Flow Network (Sankey Diagram)")
print("=" * 50)
print("This interactive diagram shows how materials flow between processes.")
print("Use the controls to filter by year, element, and minimum flow value.")

if mfa_system_with_results is not None:
    plotting.plot_interactive_sankey(mfa_system_with_results)
else:
    print("❌ No deterministic results available for Sankey diagram")

🌊 Material Flow Network (Sankey Diagram)
This interactive diagram shows how materials flow between processes.
Use the controls to filter by year, element, and minimum flow value.


interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'link': {'source': [0, 1, 2, 3, 3, 4, 4, 5, 6, 7, 7, 5, 8, 5, 9, 9,
                                  10, 10],
                       'target': [2, 2, 3, 4, 5, 0, 1, 6, 7, 0, 1, 8, 10, 9, 0, 1,
                                  0, 1],
                       'value': [100.0, 100.0, 200.0, 100.0, 100.0, 50.0, 50.0,
                                 40.0, 40.0, 20.0, 20.0, 30.0, 30.0, 30.0, 15.0,
                                 15.0, 8.133, 0.0]},
              'node': {'color': 'blue',
                       'label': [Atmosphere, Environment, Cultivation, Harvest,
                                 Grain Processing & Consumption, Straw d&C,
                                 Utilization in construction, Incineration,
                                 Incorporation, Animal bedding, Lithosphere]},
              'type': 'sankey',
              'uid': '8be6043d-de83-41c8-b1a4-f0b2ec2153c8'}],
    'layout': {'font': {'size': 12},
               'height': 700,
         

In [ ]:
# ==============================================================================
# CELL 11: STOCK EVOLUTION OVERVIEW
# ==============================================================================
"""
Copy this entire section into your eleventh Jupyter cell:

## 📊 Stock Evolution Overview

**Purpose:** Get a comprehensive view of all stock dynamics in the system.
**Features:**
- Total system stock evolution
- Individual stock breakdown
- Color-coded by process type
- Interactive element selection

**Use cases:**
- Understand overall system stock dynamics
- Compare stock evolution between elements
- Identify processes with significant stock changes
- Track long-term material accumulation trends
"""

# Stock Evolution Analysis
print("📊 Stock Evolution Overview")
print("=" * 50)

if mfa_system_with_results is not None:
    # Overall stock evolution
    print("\n📈 Overall Stock Evolution")
    plotting.plot_stock_evolution(mfa_system_with_results, dsm_params, fomp_params)
else:
    print("❌ No deterministic results available for stock evolution analysis")

📊 Stock Evolution Overview

📈 Overall Stock Evolution


interactive(children=(Dropdown(description='Element:', options=('material', 'WC', 'DM', 'CC'), value='material…

FigureWidget({
    'data': [{'line': {'color': '#d62728', 'width': 3},
              'mode': 'lines',
              'name': 'Total Stock (MATERIAL)',
              'type': 'scatter',
              'uid': '3c07057e-6602-4f38-9f16-a58c5f4ab27d',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'y': array([ 0.00000000e+00, -3.55271368e-15,  0.00000000e+00,  0.00000000e+00,
                           0.00000000e+00,  0.00000000e+00, -2.84217094e-14, -2.84217094e-14,
                          -2.84217094e-14, -2.84217094e-14, -5.68434189e-14, -5.68434189e-14,
                           0.00000000e+00,  0.00000000e+00,  5.68434189e-14,  0.00000000e+00,
                           0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
                           0.00000000e+00,  0.00000000e+00,  0.000000

In [ ]:
# ==============================================================================
# CELL 12: SYSTEM EFFICIENCY METRICS
# ==============================================================================
"""
Copy this entire section into your twelfth Jupyter cell:

## 🔍 System Efficiency Metrics

**Purpose:** Analyze system performance through key efficiency indicators.
**Metrics:**
- **Recycling Rate:** Percentage of internal material flows
- **Recovery Rate:** Ratio of outputs to inputs
- **Material Efficiency:** Useful output per unit input

**Use cases:**
- Assess system circularity
- Compare efficiency between scenarios
- Identify improvement opportunities
- Track efficiency trends over time
"""

# System Efficiency Analysis
print("🔍 System Efficiency Metrics")
print("=" * 50)

if mfa_system_with_results is not None:
    plotting.plot_system_efficiency_metrics(mfa_system_with_results)
else:
    print("❌ No deterministic results available for efficiency analysis")

🔍 System Efficiency Metrics


interactive(children=(Dropdown(description='Element:', options=('material', 'WC', 'DM', 'CC'), value='material…

FigureWidget({
    'data': [{'line': {'color': '#1f77b4', 'width': 3},
              'mode': 'lines+markers',
              'name': 'Recycling Rate (%)',
              'type': 'scatter',
              'uid': 'fbe004b2-5c12-4908-a664-ff98160a9f6d',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'y': [79.63017846652315, 79.58846105312784, 79.55086595225194,
                    79.51650711014409, 79.48475056080908, 79.45513026222793,
                    79.42729555472286, 79.40097719661765, 79.37596474014438,
                    79.35209106274908, 79.32922154550768, 73.35737808858642,
                    79.32747572163142, 79.30422079963881, 79.28188982455679,
                    79.26039667579734, 79.23966788683222, 79.2196403132076,
                    79.20025930076996, 79.18147723315974, 79.1632523699364

In [ ]:
# ==============================================================================
# CELL 13: PROCESS DYNAMICS (INFLOW-STOCK-OUTFLOW)
# ==============================================================================
"""
Copy this entire section into your thirteenth Jupyter cell:

## 📈 Process Dynamics (Inflow-Stock-Outflow)

**Purpose:** Analyze the complete dynamics of individual processes.
**Features:**
- Side-by-side view of inflow, stock, and outflow
- Process-specific analysis
- Element selection
- Smart titles based on process type

**Use cases:**
- Understand process behavior in detail
- Identify process bottlenecks
- Analyze process efficiency
- Compare process dynamics across the system
"""

# Process Dynamics
print("📈 Process Dynamics (Inflow-Stock-Outflow)")
print("=" * 50)
print("These plots show inflow, stock, and outflow dynamics for selected processes.")

if mfa_system_with_results is not None:
    plotting.plot_process_dynamics(
        mfa_system_with_results, all_excel_data["2_1_Definition_Processes"]
    )
else:
    print("❌ No deterministic results available for process dynamics")

📈 Process Dynamics (Inflow-Stock-Outflow)
These plots show inflow, stock, and outflow dynamics for selected processes.


In [ ]:
# --- Plot all defined stocks as bar charts for a selected year and element ---

from src.plotting import plot_stock_bars_by_year

# Make sure you have run the MFA calculation and have mfa_system_results available
plot_stock_bars_by_year(mfa_system_with_results)

interactive(children=(Dropdown(description='Year:', options=(2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2…

FigureWidget({
    'data': [{'marker': {'color': ['#2ca02c', '#2ca02c', '#2ca02c', '#2ca02c']},
              'type': 'bar',
              'uid': 'f431a0dd-413e-4185-9b0b-ef06cc41c4a8',
              'x': [Atmosphere, Environment, Utilization in construction,
                    Lithosphere],
              'y': [0.0, 0.0, 0.0, 0.0]}],
    'layout': {'height': 500,
               'shapes': [{'line': {'color': 'black', 'width': 2},
                           'type': 'line',
                           'x0': -0.5,
                           'x1': 3.5,
                           'y0': 0,
                           'y1': 0}],
               'template': '...',
               'title': {'text': 'Stock Values by Process for MATERIAL in 2025'},
               'xaxis': {'title': {'text': 'Process'}},
               'yaxis': {'title': {'text': 'Stock in Mg (can be positive or negative)'}}}
})

In [35]:
# ==============================================================================
# CELL 14: DSM STOCK DETAILS (IF APPLICABLE)
# ==============================================================================
"""
Copy this entire section into your fourteenth Jupyter cell:

## 🔄 DSM Stock Details (Dynamic Stock Model)

**Purpose:** Detailed analysis of DSM processes showing stock composition over time.
**Features:**
- Initial stock decay visualization
- New stock accumulation by category
- Total stock evolution
- Element-specific analysis

**Use cases:**
- Understand DSM process behavior
- Analyze stock turnover rates
- Track material aging in stocks
- Compare different DSM processes
"""

# DSM Stock Details (if DSM processes exist)
print("🔄 DSM Stock Details")
print("=" * 50)

if mfa_system_with_results is not None and dsm_params and dsm_details:
    plotting.plot_dsm_stock_details(mfa_system_with_results, dsm_params, dsm_details)
else:
    print("ℹ️  No DSM processes found or no detailed results available")

🔄 DSM Stock Details
ℹ️  No DSM processes found or no detailed results available


In [36]:
# ==============================================================================
# CELL 15: FOMP STOCK DETAILS (IF APPLICABLE)
# ==============================================================================
"""
Copy this entire section into your fifteenth Jupyter cell:

## 🌱 FOMP Stock Details (First-Order Mineralization Process)

**Purpose:** Detailed analysis of FOMP processes showing organic matter dynamics.
**Features:**
- Organic matter stock evolution
- Cumulative input tracking
- Mineralization rate analysis
- Element-specific analysis

**Use cases:**
- Understand organic matter decomposition
- Analyze mineralization rates
- Track carbon sequestration
- Compare different FOMP processes
"""

# FOMP Stock Details (if FOMP processes exist)
print("🌱 FOMP Stock Details")
print("=" * 50)

if mfa_system_with_results is not None and fomp_params:
    plotting.plot_fomp_stock_details(mfa_system_with_results, fomp_params)
else:
    print("ℹ️  No FOMP processes found")

🌱 FOMP Stock Details


interactive(children=(Dropdown(description='FOMP Process:', options=(10,), value=10), Dropdown(description='El…

FigureWidget({
    'data': [{'line': {'color': '#2ca02c', 'width': 3},
              'mode': 'lines',
              'name': 'Organic Matter Stock',
              'type': 'scatter',
              'uid': 'fa1c140e-846d-435e-8b49-7552d5bca59a',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'y': array([  0.        ,  21.867     ,  45.374025  ,  70.48007438,  97.14517252,
                          125.3303432 , 154.99758462, 186.10984501, 218.63099888, 252.52582391,
                          287.75997831, 324.29997885, 318.37917938, 358.5270999 , 399.8580224 ,
                          442.34237184, 485.95131254, 530.65672973, 576.43121149, 623.2480312 ,
                          671.08113042, 719.90510216, 769.69517461, 820.42719524, 872.07761536,
                          924.62347498])},
             {'lin

In [ ]:
# ==============================================================================
# CELL 16: SUMMARY DASHBOARD
# ==============================================================================
"""
Copy this entire section into your sixteenth Jupyter cell:

## 📋 Summary Dashboard

**Purpose:** Comprehensive overview of key system indicators and KPIs.
**Features:**
- Multi-panel dashboard layout
- Total stock evolution for all elements
- System flows overview
- Process type distribution
- Key metrics gauge

**Use cases:**
- Quick system overview
- Presentation to stakeholders
- System status monitoring
- Comparison between scenarios
"""

# Summary Dashboard
print("📋 Summary Dashboard")
print("=" * 50)

if mfa_system_with_results is not None:
    plotting.plot_summary_dashboard(mfa_system_with_results, dsm_params, fomp_params)
else:
    print("❌ No deterministic results available for dashboard")

📋 Summary Dashboard


In [38]:
# ==============================================================================
# CELL 17: SCENARIO MANAGEMENT
# ==============================================================================
"""
Copy this entire section into your seventeenth Jupyter cell:

## 🎯 Scenario Management

**Purpose:** Save, load, and compare different parameter configurations.
**Features:**
- Save current scenario configuration
- Load existing scenarios
- Create alternative scenarios
- Scenario comparison tools

**Use cases:**
- Compare different policy scenarios
- Sensitivity analysis
- Parameter optimization
- Scenario archiving
"""

# Scenario Management
print("🎯 Scenario Management")
print("=" * 50)

# Initialize scenario manager
scenario_manager = utils.ScenarioManager()

# Save current scenario
current_scenario_name = "baseline_scenario"
scenario_manager.save_scenario(
    current_scenario_name,
    config,
    description="Baseline scenario with current parameters",
)

# List available scenarios
print("\n📋 Available Scenarios:")
scenarios = scenario_manager.list_scenarios()
for scenario in scenarios:
    print(f"   - {scenario['name']}: {scenario['description']}")

# Example: Create alternative scenarios
print("\n🔄 Creating Alternative Scenarios...")

# Scenario 1: High recycling
config_high_recycling = utils.create_config_from_scenario(
    AnalysisConfig, scenario_manager.load_scenario(current_scenario_name)
)
config_high_recycling.mc_iterations = 50  # Reduce MC iterations for faster testing
scenario_manager.save_scenario(
    "high_recycling", config_high_recycling, description="High recycling rate scenario"
)

# Scenario 2: Extended time horizon
config_extended = utils.create_config_from_scenario(
    AnalysisConfig, scenario_manager.load_scenario(current_scenario_name)
)
config_extended.end_year = 2060
scenario_manager.save_scenario(
    "extended_horizon", config_extended, description="Extended time horizon to 2060"
)

print("✅ Alternative scenarios created successfully!")

🎯 Scenario Management
✅ Scenario 'baseline_scenario' saved successfully.

📋 Available Scenarios:
   - baseline_scenario: Baseline scenario with current parameters
   - extended_horizon: Extended time horizon to 2060
   - high_recycling: High recycling rate scenario

🔄 Creating Alternative Scenarios...
✅ Scenario 'high_recycling' saved successfully.
✅ Scenario 'extended_horizon' saved successfully.
✅ Alternative scenarios created successfully!


In [ ]:
# ==============================================================================
# CELL 18: EXPORT RESULTS
# ==============================================================================
"""
Copy this entire section into your eighteenth Jupyter cell:

## 💾 Export Results

**Purpose:** Save analysis results to files for further analysis and reporting.
**Features:**
- Excel export with multiple sheets
- Comprehensive data export
- Monte Carlo results export
- Summary statistics

**Use cases:**
- Further analysis in Excel
- Report generation
- Data archiving
- Sharing results with stakeholders
"""

# Export results
print("💾 Exporting Results")
print("=" * 30)

# Create output directory if it doesn't exist
output_dir = os.path.dirname(config.output_path)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"📁 Created output directory: {output_dir}")

# Export deterministic results
if mfa_system_with_results is not None:
    utils.export_results_to_excel(mfa_system_with_results, config.output_path)
    print(f"✅ Results exported to: {config.output_path}")

    # Show what was exported
    print("\n📊 Exported data includes:")
    print(f"   - Flows time series ({len(mfa_system_with_results.FlowDict)} flows)")
    print(f"   - Stocks time series ({len(mfa_system_with_results.StockDict)} stocks)")
    print(f"   - Time range: {config.start_year} - {config.end_year}")
    print(f"   - Elements: {', '.join(config.elements)}")

# Export Monte Carlo results
if config.run_monte_carlo and df_mc_results is not None:
    mc_output_path = config.output_path.replace(".xlsx", "_MonteCarlo.xlsx")
    with pd.ExcelWriter(mc_output_path) as writer:
        df_mc_results.to_excel(writer, sheet_name="MC_Results", index=False)

        # Add summary statistics
        if "final_C_stock_soil" in df_mc_results.columns:
            summary_stats = df_mc_results["final_C_stock_soil"].describe()
            summary_stats.to_frame("final_C_stock_soil").to_excel(
                writer, sheet_name="Summary_Stats"
            )

    print(f"✅ Monte Carlo results exported to: {mc_output_path}")
    print(f"   - {len(df_mc_results)} simulation runs")
    print(f"   - {len(df_mc_results.columns)} parameters tracked")

print("\n🎉 Analysis complete!")
print("You can now:")
print("  - Review the interactive plots above")
print("  - Open the exported Excel files for detailed data")
print("  - Use scenario management to compare different configurations")
print("  - Modify the configuration and re-run for different scenarios")

# ==============================================================================
# APPENDIX: Publication-Ready Sankey Diagram Enhancements (for Jupyter)
# ==============================================================================
"""
Copy these code snippets into your Jupyter cells as needed to enhance your Sankey diagrams for publication.
"""

# --- Export Button for Plotly Figures (SVG/PNG) ---
import ipywidgets as widgets


def show_export_buttons(fig, filename_base="sankey_diagram"):
    def export_svg(_):
        fig.write_image(f"{filename_base}.svg")
        print(f"Exported as {filename_base}.svg")

    def export_png(_):
        fig.write_image(f"{filename_base}.png")
        print(f"Exported as {filename_base}.png")

    btn_svg = widgets.Button(description="Export as SVG")
    btn_png = widgets.Button(description="Export as PNG")
    btn_svg.on_click(export_svg)
    btn_png.on_click(export_png)
    display(widgets.HBox([btn_svg, btn_png]))
    fig.show()


# Usage example (after creating your Sankey fig):
# show_export_buttons(fig)

# --- Example Color Mapping for Sankey Nodes/Links by Type ---
# Define your process types and assign colors
process_types = ["DSM", "FOMP", "Regular"]
type_colors = {
    "DSM": "#ff7f0e",  # Orange
    "FOMP": "#2ca02c",  # Green
    "Regular": "#1f77b4",  # Blue
}
# Suppose you have a list of node types in node_types
# node_colors = [type_colors.get(t, '#cccccc') for t in node_types]
# fig.update_traces(node=dict(color=node_colors))

# --- Markdown Best Practices for Publication Graphics ---
"""
**Best Practices for Publication-Ready Sankey Diagrams:**
- Use SVG export for vector quality.
- Set font to Arial or Times New Roman, size 12+.
- Use high-contrast colors for clarity.
- Remove unnecessary gridlines and backgrounds.
- Label nodes and flows clearly.
"""

💾 Exporting Results
--> Exporting results to 'data/02_output/results.xlsx'...
✅ Results exported to: data/02_output/results.xlsx

📊 Exported data includes:
   - Flows time series (18 flows)
   - Stocks time series (8 stocks)
   - Time range: 2025 - 2050
   - Elements: material, WC, DM, CC

🎉 Analysis complete!
You can now:
  - Review the interactive plots above
  - Open the exported Excel files for detailed data
  - Use scenario management to compare different configurations
  - Modify the configuration and re-run for different scenarios


'\n**Best Practices for Publication-Ready Sankey Diagrams:**\n- Use SVG export for vector quality.\n- Set font to Arial or Times New Roman, size 12+.\n- Use high-contrast colors for clarity.\n- Remove unnecessary gridlines and backgrounds.\n- Label nodes and flows clearly.\n'